In [2]:
from dataclasses import dataclass


sample_line = "[.##.] (3) (1,3) (2) (2,3) (0,2) (0,1) {3,5,4,7}"

@dataclass
class Entry:
    target_lights: list[str]
    buttons: list[tuple[int]]
    last_part: list[int]
    current_lights: list[str]

    def __init__(self, line):
        end_first_part = line.find(" ")
        beginning_last_part = line.find("{")
        self.target_lights = list(line[1:end_first_part-1])
        self.buttons = [[int(value) for value in button[1:-1].split(',')] for button in line[end_first_part+1:beginning_last_part - 1].split(" ")]
        self.last_part = [int(value) for value in line[beginning_last_part+1:-1].split(',')]
        self.current_lights = ["."] * len(self.target_lights)

    def press_button(self, button: list[int], current_lights):
        for i in button:
            if current_lights[i] == ".":
                current_lights[i] = "#"
            else:
                current_lights[i] = "."
        return current_lights

    

In [23]:
line1 = Entry(sample_line)

In [24]:
line1

Entry(target_lights=['.', '#', '#', '.'], buttons=[[3], [1, 3], [2], [2, 3], [0, 2], [0, 1]], last_part=[3, 5, 4, 7], current_lights=['.', '.', '.', '.'])

In [30]:
print(line1.current_lights)
line1.press_button([1, 3])
line1

['.', '.', '.', '.']


Entry(target_lights=['.', '#', '#', '.'], buttons=[[3], [1, 3], [2], [2, 3], [0, 2], [0, 1]], last_part=[3, 5, 4, 7], current_lights=['.', '#', '.', '#'])

In [31]:
import random

In [19]:
factor = 10

# 20x1   20x5x1   20x5x5x1    2500x4 ...

sample_line2 = "[...#.] (0,2,3,4) (2,3) (0,4) (0,1,2) (1,2,3,4) {7,5,12,7,2}"
sample_line3 = "[.###.#] (0,1,2,3,4) (0,3,4) (0,1,2,4,5) (1,2) {10,11,11,5,10,5}"

input_line1 = "[#..#.] (1,2,4) (2) (0,4) (1,2,3,4) (0,1,3) (1,2,3) (0,1,3,4) {19,24,34,11,36}"
input_line2 = "[.#.#.####.] (7,8) (3,5,9) (0,2,6,8) (2,3,4,9) (2,3,8) (0,1,2,3,4,6,8,9) (2,5) (7) (0,5,6,7) (1,2,3,4,5,9) (0,1,4,5,8) (0,1,3,5,7,8) {48,25,64,55,33,73,34,35,50,40}"

shortest = 5_000
found_solution = False

# try 50 times with 1 button
# try 500 times with 2 buttons
# try 5000 times with 3 buttons
# ...
# random approach didn't work for some lines

for i in range(1, 6):
    if found_solution:
        break
    button_choices = []
    print(5 * factor**i)

    for j in range(5 * factor**i):
        line1 = Entry(input_line1)
        button_choices = []
        for k in range(0, i):
            button = random.choice(line1.buttons)
            button_choices.append(button[0])
            # print(button)
            line1.press_button(button)
        # print(line1.current_lights)


        if(line1.target_lights == line1.current_lights):
            print(f"found solution: {button_choices}")
            if len(button_choices) < shortest:
                shortest = len(button_choices)
                found_solution = True
            break
    

50


NameError: name 'random' is not defined

Try recursive approach that goes through each button press, then the button twice and combined with all others ...

FAILED

In [61]:
def press_buttons(entry: Entry, current_lights, prev_buttons: list, button_num, target_lights):
    current_lights = entry.press_button(entry.buttons[button_num], current_lights)
    if current_lights == target_lights:
        return prev_buttons.append(button_num)

    else:
        for i in range(len(entry.buttons)):
            # print(prev_buttons)
            prev_buttons.append(button_num)
            return press_buttons(entry, current_lights, prev_buttons, i, target_lights)
    
entry1 = Entry(sample_line)
press_buttons(entry1, ["."] * len(entry1.target_lights), [], 0, entry1.target_lights)

RecursionError: maximum recursion depth exceeded

OR-Tools approach - abandoned, could have worked https://developers.google.com/optimization/mip

In [40]:
from ortools.init.python import init
from ortools.linear_solver import pywraplp

# Create the linear solver with the GLOP backend.
solver = pywraplp.Solver.CreateSolver("SCIP")
if not solver:
    print("Could not create solver SCIP")


line1 = Entry(sample_line)
print(line1)

max_num = 200

vars = []
for i, button in enumerate(line1.buttons):
    var = solver.IntVar(0, max_num, "button_" + str(i))
    vars.append(var)

print("Number of variables =", solver.NumVariables())

# Objective terms? (or constraints)
# the (button presses % 2) must add up to target_lights
# objective_terms = []
# for i in range(line1.target_lights):
#     for j, button in enumerate(line1.buttons):
#         if i in button:
            


# solver.Minimize()



Entry(target_lights=['.', '#', '#', '.'], buttons=[[3], [1, 3], [2], [2, 3], [0, 2], [0, 1]], last_part=[3, 5, 4, 7], current_lights=['.', '.', '.', '.'])
Number of variables = 6


In [36]:
from itertools import combinations_with_replacement
from pathlib import Path


def check_line(line: Entry) -> int:
    result_not_found = True
    counter = 0

    while result_not_found:
        counter += 1

        line1 = Entry(sample_line3)

        for combination in combinations_with_replacement(line.buttons, counter):
            current_lights = ["."] * len(line.target_lights)
            for button in combination:
                current_lights = line.press_button(button, current_lights)
                if current_lights == line.target_lights:
                    # print(combination)
                    result_not_found = False
                    # print(combination)
                    return len(combination)


files = ['sample', 'input']
path = "day10"

for file in files:
    sum = 0
    with Path(file).open() as input:
        for line in input.readlines():
            entry = Entry(line.strip())
            sum += check_line(entry)
    
    print(f"{file}: {sum}")



sample: 7
input: 473


# Part 2

Now I want to use [Google's OR Tools](https://developers.google.com/optimization) and it seems right for the job (and similar to [one of the examples](https://developers.google.com/optimization/assignment/assignment_example)?).

In [27]:
from ortools.init.python import init
from ortools.linear_solver import pywraplp
from pathlib import Path

# Create the linear solver with the GLOP backend.
solver = pywraplp.Solver.CreateSolver("SCIP")
if not solver:
    print("Could not create solver SCIP")


def solve_line_2(line:str) -> int:
    line = Entry(line)
    # print(line)

    max_num = 200

    vars = []
    for i, button in enumerate(line.buttons):
        var = solver.IntVar(0, max_num, "button_" + str(i))
        vars.append(var)

    cols = {key: [] for key in range(len(line.last_part))}
    # print(cols)

    # create the constraints
    for i, button in enumerate(line.buttons):
        for j in button:
            cols[j].append(i)

    for key, value in cols.items():
        solver.Add(solver.Sum([vars[i] for i in value]) == line.last_part[key])
        # print(f"{[vars[i] for i in value] =}, {line.last_part[key] =}")


    # print("Number of variables =", solver.NumVariables())
    objective_terms = []
    for var in vars:
        objective_terms.append(var)
    solver.Minimize(solver.Sum(objective_terms))

    # print(f"Solving with {solver.SolverVersion()}")
    status = solver.Solve()

    overall = 0

    if status == pywraplp.Solver.OPTIMAL or status == pywraplp.Solver.FEASIBLE:
        # print(f"Total cost = {solver.Objective().Value()}\n")
        for i, var in enumerate(vars):
            # print(f"Button {i}: {var.solution_value()}")
            overall += var.solution_value()
        
        # print(f"Presses for this pattern: {int(overall)}")
    else:
        print("No solution found.")

    return overall


files = ['sample', 'input']
path = "day10"

for file in files:
    sum = 0
    with Path(file).open() as input:
        for line in input.readlines():
            sum += solve_line_2(line.strip())
    
    print(f"{file}: {int(sum)}")

sample: 33
input: 18681
